# Day 078 — Exercise 4: synthesize_speech and narrate_image

**What you'll build:** The TTS pipeline and the first cross-modal pipeline (image → describe → speak).

**Why it matters:** `narrate_image` is a two-stage pipeline that produces audio narration of any image. Text is the bridge: vision produces it, TTS consumes it.

In [ ]:
from PIL import Image as _PILImage

def _make_mock_image(w=100, h=100, color=(100, 150, 200)):
    return _PILImage.new('RGB', (w, h), color=color)

_mock_describe_fn   = lambda img, q: 'A test image with a solid color background.'
_mock_transcribe_fn = lambda src: {'text': 'Hello world.', 'segments': [
    {'start': 0.0, 'end': 1.0, 'text': 'Hello world.'}]}
_mock_tts_fn        = lambda text, voice, rate, pitch: b'AUDIO:' + text[:8].encode()
from pathlib import Path

MEDIA_EXTENSIONS = {
    'image': {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp', '.tiff'},
    'audio': {'.mp3', '.wav', '.ogg', '.flac', '.m4a', '.aac'},
    'video': {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv'},
}

def detect_media_type(path):
    ext = Path(path).suffix.lower()
    for media_type, extensions in MEDIA_EXTENSIONS.items():
        if ext in extensions:
            return media_type
    return 'unknown'
import io, base64

def describe_media(source, describe_fn=None):
    from PIL import Image
    if isinstance(source, (str, Path)):
        image = Image.open(source)
    else:
        image = source
    prompt = 'Describe this image in detail, including all visible content.'
    if describe_fn is not None:
        return describe_fn(image, prompt)
    import ollama
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': prompt, 'images': [img_b64]}],
    )
    return resp['message']['content']
def transcribe_media(source, transcribe_fn=None):
    if transcribe_fn is not None:
        return transcribe_fn(source)
    import whisper, os, tempfile
    model = whisper.load_model('base')
    if isinstance(source, bytes):
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
            f.write(source)
            tmp = f.name
        try:
            raw = model.transcribe(tmp)
        finally:
            os.unlink(tmp)
    else:
        raw = model.transcribe(str(source))
    segments = [
        {'start': s['start'], 'end': s['end'], 'text': s['text'].strip()}
        for s in raw.get('segments', [])
    ]
    return {'text': raw.get('text', '').strip(), 'segments': segments}


## Task

1. `synthesize_speech(text, voice='en-US-AriaNeural', rate='+0%', pitch='+0Hz', tts_fn=None) -> bytes`
   - If `tts_fn`: `return tts_fn(text, voice, rate, pitch)`
   - Else: `import edge_tts; async def _run(): collect audio chunks; return b''.join(chunks); return asyncio.run(_run())`

2. `narrate_image(source, voice='en-US-AriaNeural', describe_fn=None, tts_fn=None) -> dict`
   - `description = describe_media(source, describe_fn=describe_fn)`
   - `audio = synthesize_speech(description, voice=voice, tts_fn=tts_fn)`
   - `return {'description': description, 'audio': audio}`

## Your Implementation

In [ ]:
import asyncio

def synthesize_speech(text, voice='en-US-AriaNeural', rate='+0%', pitch='+0Hz',
                      tts_fn=None):
    """Convert text to speech. Returns MP3 bytes."""
    raise NotImplementedError

def narrate_image(source, voice='en-US-AriaNeural', describe_fn=None, tts_fn=None):
    """Describe an image then speak the description.
    Returns dict with keys: description (str), audio (bytes).
    """
    raise NotImplementedError


In [ ]:
import asyncio

def synthesize_speech(text, voice='en-US-AriaNeural', rate='+0%', pitch='+0Hz',
                      tts_fn=None):
    if tts_fn is not None:
        return tts_fn(text, voice, rate, pitch)
    import edge_tts
    async def _run():
        chunks = []
        async for chunk in edge_tts.Communicate(
                text, voice, rate=rate, pitch=pitch).stream():
            if chunk['type'] == 'audio':
                chunks.append(chunk['data'])
        return b''.join(chunks)
    return asyncio.run(_run())

def narrate_image(source, voice='en-US-AriaNeural', describe_fn=None, tts_fn=None):
    description = describe_media(source, describe_fn=describe_fn)
    audio = synthesize_speech(description, voice=voice, tts_fn=tts_fn)
    return {'description': description, 'audio': audio}


## Automated checks

In [ ]:

score, total = 0, 5
try:
    audio = synthesize_speech('Hello', tts_fn=_mock_tts_fn)
    assert isinstance(audio, bytes) and len(audio) > 0
    score += 1; print("✅ synthesize_speech returns bytes")

    calls = []
    def _tfn(text, voice, rate, pitch): calls.append((text,voice,rate,pitch)); return b'A'
    synthesize_speech('Test text', voice='en-GB-RyanNeural',
                      rate='+10%', pitch='+5Hz', tts_fn=_tfn)
    assert calls[0] == ('Test text', 'en-GB-RyanNeural', '+10%', '+5Hz')
    score += 1; print("✅ tts_fn receives (text, voice, rate, pitch)")

    img = _make_mock_image()
    result = narrate_image(img, describe_fn=_mock_describe_fn, tts_fn=_mock_tts_fn)
    assert isinstance(result, dict)
    score += 1; print("✅ narrate_image returns dict")

    assert 'description' in result and 'audio' in result
    score += 1; print("✅ result has 'description' and 'audio'")

    assert isinstance(result['description'], str) and isinstance(result['audio'], bytes)
    score += 1; print("✅ description is str, audio is bytes")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
import asyncio

def synthesize_speech(text, voice='en-US-AriaNeural', rate='+0%', pitch='+0Hz',
                      tts_fn=None):
    if tts_fn is not None:
        return tts_fn(text, voice, rate, pitch)
    import edge_tts
    async def _run():
        chunks = []
        async for chunk in edge_tts.Communicate(
                text, voice, rate=rate, pitch=pitch).stream():
            if chunk['type'] == 'audio':
                chunks.append(chunk['data'])
        return b''.join(chunks)
    return asyncio.run(_run())

def narrate_image(source, voice='en-US-AriaNeural', describe_fn=None, tts_fn=None):
    description = describe_media(source, describe_fn=describe_fn)
    audio = synthesize_speech(description, voice=voice, tts_fn=tts_fn)
    return {'description': description, 'audio': audio}
```

**Why return both `description` and `audio` in narrate_image?** The text description is useful on its own — callers can display it, log it, or store it. Returning only the audio would discard intermediate output that took a full LLM call to produce.

</details>